### Data Preparation for Both Datasets


In [1]:
import torch

# 1. Check if the binary was compiled with CUDA support
print(f"Is CUDA compiled in? {torch.backends.cuda.is_built()}")

# 2. Check if the system can actually see your GPU
print(f"Is GPU available? {torch.cuda.is_available()}")

# 3. Print the specific CUDA version PyTorch is using
if torch.cuda.is_available():
    print(f"Using CUDA version: {torch.version.cuda}")
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

Is CUDA compiled in? True
Is GPU available? True
Using CUDA version: 12.4
Device Name: Quadro P620


In [2]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from typing import List, Dict, Optional

##### Step 1
We proceed to:
1. Remove triples duplicates, i.e. (h,r,t) that appear multiple times
2. Remove triples with NAN/Null entries
3. Remove non-valid relations (or predicates). N.W. We do not remove "invalid nodes" as a non-valid node does not end up in the merged df because is not associable with a node type
4. Remove singleton edges, i.e. relations that appear in only one triple
5. Create a merged df that connects node types with each triple, excluding both nodes that do not participate in any triple and nodes that are in triple but not listed in `nodes.csv`

In [4]:
NODES_PATH = 'raw_data/nodes_filtered.csv'
EDGES_PATH = 'raw_data/edges_filtered.csv'
INVALID_RELS = ["Null", "None", "N/A", "", " ", "none", "null"]

# csv --> DataFrame
nodes = pd.read_csv(NODES_PATH)
edges = pd.read_csv(EDGES_PATH)
article_nodes = set(nodes.loc[nodes['type'] == 'Article', 'name']) #set to reduce lookup time

# cleaned and merged DataFrame
merged_df = (
    edges
    .query("subject not in @article_nodes and object not in @article_nodes") #remove triples with Article nodes
    .drop_duplicates() #drop duplicate triples
    .dropna(subset=["subject", "predicate", "object"]) #drop triples with missing values
    .query("predicate not in @INVALID_RELS") #drop invalid relations
    .groupby("predicate").filter(lambda x: len(x) > 1)  # Removes singleton edge types
    # First Merge: Subject Node Type
    .merge(nodes, left_on='subject', right_on='name')
    .rename(columns={'type': 'subject_type'})
    .drop(columns=['name'])
    # Second Merge: Object Node Type
    .merge(nodes, left_on='object', right_on='name', suffixes=('', '_target'))
    .rename(columns={'type': 'object_type'})
    .drop(columns=['name']) 
)

print(f"Processing complete. Final Merged Shape: {merged_df.shape}")
print(f"{merged_df[(merged_df['subject'] == 'Article') | (merged_df['object'] == 'Article')].shape[0]} triples with 'Article' nodes.")
merged_df.sample(5)

Processing complete. Final Merged Shape: (2707930, 5)
0 triples with 'Article' nodes.


,subject,predicate,object,subject_type,object_type
1216264,<http://purl.obolibrary.org/obo/HP_0003828>,Phenotype of,<http://purl.obolibrary.org/obo/MONDO_0010308>,Phenotype,Disease
1665434,<http://purl.obolibrary.org/obo/PR_O14556>,Molecularly interacts with,<http://purl.obolibrary.org/obo/PR_P04406>,Protein,Protein
1089643,<http://purl.obolibrary.org/obo/GO_0098978>,Location of,<http://purl.obolibrary.org/obo/PR_Q9NPH3>,GO,Protein
985157,<http://purl.obolibrary.org/obo/PR_Q9UKT8>,Molecularly interacts with,<http://purl.obolibrary.org/obo/PR_Q9UM13>,Protein,Protein
2077198,<http://purl.obolibrary.org/obo/GO_0005815>,Location of,<http://purl.obolibrary.org/obo/PR_Q02750>,GO,Protein


#### Step 2

We want each `SubclassOf` predicate to be domain specific. 

This type of relation happens only between nodes of the same type and describes taxonomical/hierarchical connectivity. We want to avoid using a general `SubclassOf` for all different taxonomies, and instead create specific ones s.a. `SubclassOf_Phenotype`

In [5]:
merged_df[merged_df['predicate'].str.contains("Subclassof")][['subject_type', 'object_type']].drop_duplicates()

,subject_type,object_type
3,Disease,Disease
5,Gene,Genomic feature
6,GO,GO
25,Phenotype,Phenotype
61,Protein,Protein
276,Genomic feature,Genomic feature
10704,Gene,Gene
15898,Protein,GO
88489,Protein,Genomic feature
202242,Disease,Phenotype


In [6]:
merged_df = merged_df.assign(
        predicate=lambda df: df.apply(
            lambda row: f"Subclassof_{row['subject_type']}_to_{row['object_type']}" 
            if row['predicate'] == 'Subclassof' #and row['subject_type'] == row['object_type'] 
            else row['predicate'], axis=1
        )
)

#check no pure Subclassof relations remain
print(f"Number of 'pure' Subclassof relations: {len(merged_df[merged_df['predicate'] == 'Subclassof'])}")

#show one sample for each modified Subclassof relations
merged_df[merged_df['predicate'].str.contains('Subclassof')].drop(columns=['subject', 'object']).groupby('predicate').sample(1)

Number of 'pure' Subclassof relations: 0


,predicate,subject_type,object_type
362442,Subclassof_Disease_to_Disease,Disease,Disease
202242,Subclassof_Disease_to_Phenotype,Disease,Phenotype
490161,Subclassof_GO_to_GO,GO,GO
1588630,Subclassof_Gene_to_Gene,Gene,Gene
454400,Subclassof_Gene_to_Genomic feature,Gene,Genomic feature
738603,Subclassof_Genomic feature_to_Genomic feature,Genomic feature,Genomic feature
2304124,Subclassof_Genomic feature_to_Protein,Genomic feature,Protein
2572314,Subclassof_Person_to_Person,Person,Person
182311,Subclassof_Phenotype_to_Phenotype,Phenotype,Phenotype
2339736,Subclassof_Protein_to_GO,Protein,GO


##### Step 3 -- generate Full_Graph and Differential Diagnosis datasets
Recall in KGs we need to perform a **transductive** split, we cannot split randomly into train/valid/test. 

**-->** train must contain all **node istances** at least **once**.

**-->** train must also contain all **predicate types** at least **once**.

In [7]:
def generate_transductive_splits(
    df: pd.DataFrame, 
    n_splits: int = 1, 
    seed: int = 42, 
    target_triples: Optional[Dict[str, str]] = None, 
    include_valid: bool = True,
    min_group_size: int = 5,
    test_ratio: float = 0.2,   # 20% Test
    val_ratio: float = 0.1     # 10% Valid
) -> List[Dict[str, Optional[pd.DataFrame]]]:
    
    # 1. MARK PRIORITY --> targeted triples get higher priority
    df['priority'] = 0 
    if target_triples:
        # identify target triples and set priority to 1
        conditions = [ 
            f"{k}.str.contains('{v}', regex=False)" if k == 'predicate' else f"{k} == '{v}'"
            for k, v in target_triples.items()
        ]
        target_mask = df.query(" & ".join(conditions)).index
        df.loc[target_mask, 'priority'] = 1

    results = []
    
    #create n splits
    for i in range(n_splits):
        rng = np.random.default_rng(seed + i) #for reproducibility
        
        # 2. SHUFFLE & SORT
        df['random_sort'] = rng.random(len(df)) #df rows (triples) get a random float
        df_sorted = df.sort_values(['priority', 'random_sort'], ascending=[True, True])  
        
        # 3. SAFETY NET --> compute the set cover (minimum set of triples that contain all nodes)
        subj_anchor_idx = df_sorted.drop_duplicates('subject', keep='first').index
        obj_anchor_idx = df_sorted.drop_duplicates('object', keep='first').index
        anchor_idx = subj_anchor_idx.union(obj_anchor_idx)
        
        # 4. DEFINE POOL --> POOL = ALL TRIPLES - SAFETY NET
        pool_mask = ~df_sorted.index.isin(anchor_idx)
        pool_df = df_sorted.loc[pool_mask].copy()

        # 5. SELECT TEST SET 
        if target_triples:
            #extract pool of priority 1 triples (targeted ones)
            target_pool = pool_df[pool_df['priority'] == 1].copy()
            
            # target triple (sub,r,obj). 
            # Group triples by obj and count how many subj each obj is linked to  
            target_pool['group_size'] = target_pool.groupby('object')['object'].transform('count')
            target_pool['rank'] = target_pool.groupby('object').cumcount()
            
            # Calculate Dynamic Cutoff
            target_pool['n_test_needed'] = np.where(
                target_pool['group_size'] >= min_group_size,
                np.ceil(target_pool['group_size'] * test_ratio),
                0
            )
            
            test_mask_local = target_pool['rank'] < target_pool['n_test_needed']
            test_idx = target_pool.loc[test_mask_local].index
            
        else:
            # === GENERAL MODE (Random) ===
            # Simply sample from the Safe Pool (priority doesn't matter here)
            n_test = min(int(len(df) * test_ratio), len(pool_df))
            
            # Use choice on the index for speed
            if n_test > 0:
                test_idx = pd.Index(rng.choice(pool_df.index, size=n_test, replace=False))
            else:
                test_idx = pd.Index([])

        # 6. VALIDATION (Random Sample from Remaining Pool)
        val_idx = pd.Index([])
        if include_valid:
            remaining_pool_idx = pool_df.index.difference(test_idx)
            n_val = min(int(len(df) * val_ratio), len(remaining_pool_idx))
            
            if n_val > 0:
                chosen_indices = rng.choice(remaining_pool_idx, size=n_val, replace=False)
                val_idx = pd.Index(chosen_indices)

        # 7. CLEANUP & RETURN
        train_idx = df.index.difference(test_idx.union(val_idx))

        results.append({
            'split_id': i,
            'train': df.loc[train_idx].drop(columns=['priority', 'random_sort'], errors='ignore'),
            'valid': df.loc[val_idx].drop(columns=['priority', 'random_sort'], errors='ignore') if len(val_idx) > 0 else None,
            'test':  df.loc[test_idx].drop(columns=['priority', 'random_sort'], errors='ignore')
        })
        
    df.drop(columns=['priority', 'random_sort'], inplace=True, errors='ignore')
    return results

Below let's perform some checks on datasets transductivity

In [8]:
import pandas as pd

def run_split_report(split_data, name):
    print(f"=== {name.upper()} REPORT ===")
    
    for i, s in enumerate(split_data):
        print(f"\n--- Split {i} ---")
        
        train, valid, test = s.get('train'), s.get('valid'), s.get('test')
        
        if train is None or test is None:
            print("Missing train or test set in this split. Skipping.")
            continue
        
        # 1. TRANSDUCTIVITY / INDUCTIVE (NON-TRANSDUCTIVE) CHECK
        # Create unique nodes sets 
        train_nodes = set(train['subject']).union(set(train['object']))
        test_nodes = set(test['subject']).union(set(test['object']))
        valid_nodes = set(valid['subject']).union(set(valid['object'])) if valid is not None else set()
        
        # check if test or valid have nodes that are not in train
        test_new_nodes = test_nodes - train_nodes
        valid_new_nodes = valid_nodes - train_nodes
        
        test_transductive = len(test_new_nodes) == 0
        valid_transductive = len(valid_new_nodes) == 0
        
        print(f"Transductivity (Test):  {'PASSED' if test_transductive else 'FAILED (Inductive)'}")
        if not test_transductive:
            pct_test = (len(test_new_nodes) / len(test_nodes) * 100) if test_nodes else 0
            print(f"  -> Unseen Test Nodes: {len(test_new_nodes):,} ({pct_test:.2f}% of test nodes)")
            
        print(f"Transductivity (Valid): {'PASSED' if valid_transductive else 'FAILED (Inductive)'}")
        if not valid_transductive:
            pct_valid = (len(valid_new_nodes) / len(valid_nodes) * 100) if valid_nodes else 0
            print(f"  -> Unseen Valid Nodes:{len(valid_new_nodes):,} ({pct_valid:.2f}% of valid nodes)")

        # Global info for this dataset 
        total_nodes = len(train_nodes.union(test_nodes).union(valid_nodes))
        print(f"Total Unique Nodes:     {total_nodes:,}")

        # 2. LEAKAGE CHECK
        test_leak = pd.merge(train, test, on=['subject', 'predicate', 'object']).shape[0]
        valid_leak = pd.merge(train, valid, on=['subject', 'predicate', 'object']).shape[0] if valid is not None else 0
        extra_leak = pd.merge(valid, test, on=['subject', 'predicate', 'object']).shape[0] if valid is not None else 0
        
        print(f"Leakage (Train-Test):   {test_leak} triples")
        print(f"Leakage (Train-Valid):  {valid_leak} triples")
        print(f"Leakage (Valid-Test):   {extra_leak} triples")

        # 3. PREDICATE DISTRIBUTION
        test_preds = test['predicate'].unique()
        valid_preds = valid['predicate'].unique() if valid is not None else []
        train_preds = train['predicate'].unique()
        total_preds = len(set(train_preds).union(test_preds).union(valid_preds))
        
        print(f"Test Predicates:        {len(test_preds)} (Out of {total_preds} total in split)")
        print(f"Valid Predicate Count:  {len(valid_preds)} (General check)")
        
        # 4. DATASET SIZES
        valid_size = len(valid) if valid is not None else 0
        print(f"Split Sizes:            Train: {len(train):,} | Valid: {valid_size:,} | Test: {len(test):,}")
        print("-" * 35)

    print("\n")

In [9]:
# Scenario 1: General Split (80/10/10)
general_results = generate_transductive_splits(
    merged_df, 
    n_splits=1, 
    target_triples=None, 
    include_valid=True,
    test_ratio=0.1,
    val_ratio=0.1
)

run_split_report(general_results, "General Split")


# Scenario 2: Targeted Split (Test contains only 'Has disease')
targeted_results = generate_transductive_splits(
    merged_df, 
    n_splits=1, 
    target_triples={'predicate': 'Has disease'}, 
    include_valid=False 
)

run_split_report(targeted_results, "Targeted (Has Disease) Split")   

targeted_results_valid = generate_transductive_splits(
    merged_df, 
    n_splits=1, 
    target_triples={'predicate': 'Has disease'}, 
    include_valid=True 
)

run_split_report(targeted_results_valid, "Targeted (Has Disease) Split") 


=== GENERAL SPLIT REPORT ===

--- Split 0 ---
Transductivity (Test):  PASSED
Transductivity (Valid): PASSED
Total Unique Nodes:     241,789
Leakage (Train-Test):   0 triples
Leakage (Train-Valid):  0 triples
Leakage (Valid-Test):   0 triples
Test Predicates:        85 (Out of 116 total in split)
Valid Predicate Count:  85 (General check)
Split Sizes:            Train: 2,166,344 | Valid: 270,793 | Test: 270,793
-----------------------------------


=== TARGETED (HAS DISEASE) SPLIT REPORT ===

--- Split 0 ---
Transductivity (Test):  PASSED
Transductivity (Valid): PASSED
Total Unique Nodes:     241,789
Leakage (Train-Test):   0 triples
Leakage (Train-Valid):  0 triples
Leakage (Valid-Test):   0 triples
Test Predicates:        1 (Out of 116 total in split)
Valid Predicate Count:  0 (General check)
Split Sizes:            Train: 2,706,395 | Valid: 0 | Test: 1,535
-----------------------------------


=== TARGETED (HAS DISEASE) SPLIT REPORT ===

--- Split 0 ---
Transductivity (Test):  PASSED

In [10]:
save_tasks = {
    "Full_graph": general_results,
    "DD_graph": targeted_results,
    "DD_graph_with_valid": targeted_results_valid
}

base_path = Path("KGEmb/data")

# 2. Execution
for folder_name, result_list in save_tasks.items():
    
    # Check if we need sub-folders (e.g., if you did n_splits=5)
    is_multi_split = len(result_list) > 1
    
    experiment_dir = base_path / folder_name
    experiment_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"Processing: {folder_name} (Total splits: {len(result_list)})")

    for result in result_list:
        split_id = result['split_id']
        
        if is_multi_split:
            save_dir = experiment_dir / str(split_id) #e.g.Full_graph/0
            save_dir.mkdir(exist_ok=True)
        else:
            save_dir = experiment_dir


        for dataset_type in ['train', 'valid', 'test']:
            df = result.get(dataset_type)
            
            if df is not None and not df.empty:

                file_path = save_dir / dataset_type #e.g. Full_graph/train or Full_graph/0/train

                #TODO: we need to do this cleaning (remove subj and obj types) when buiding the df not here
                clean_df = df[['subject', 'predicate', 'object']]
                
                clean_df.sample(frac=1, random_state=42).reset_index(drop=True).to_csv( #shuffle when saving 
                    file_path, 
                    sep='\t', 
                    header=False, 
                    index=False
                )
                print(f"   └── Saved: {file_path}")
            else:
                # if valid is missing
                pass

    print(f"✅ {folder_name} complete.\n")

Processing: Full_graph (Total splits: 1)
   └── Saved: KGEmb\data\Full_graph\train
   └── Saved: KGEmb\data\Full_graph\valid
   └── Saved: KGEmb\data\Full_graph\test
✅ Full_graph complete.

Processing: DD_graph (Total splits: 1)
   └── Saved: KGEmb\data\DD_graph\train
   └── Saved: KGEmb\data\DD_graph\test
✅ DD_graph complete.

Processing: DD_graph_with_valid (Total splits: 1)
   └── Saved: KGEmb\data\DD_graph_with_valid\train
   └── Saved: KGEmb\data\DD_graph_with_valid\valid
   └── Saved: KGEmb\data\DD_graph_with_valid\test
✅ DD_graph_with_valid complete.

